# 📊 Exploratory Data Analysis — Bluestock MF Capstone
## Day 3: EDA with 15+ Charts

**Project:** Mutual Fund Analytics | **Dataset:** 10 CSVs, 40 Schemes, 2022–2026  
**Tools:** Python, Pandas, Matplotlib, Seaborn, Plotly  

---
### Dataset Overview
| File | Rows | Description |
|------|------|-------------|
| fund_master | 40 | Scheme metadata |
| nav_history | 46,000 | Daily NAV 2022–2026 |
| aum_by_fund_house | 90 | Quarterly AUM |
| monthly_sip_inflows | 48 | Monthly SIP data |
| category_inflows | 144 | Category-level inflows |
| industry_folio_count | 21 | Quarterly folio count |
| investor_transactions | 32,778 | Transaction records |
| portfolio_holdings | 322 | Stock holdings |
| benchmark_indices | 8,050 | NIFTY/BSE indices |
| scheme_performance | 40 | Performance metrics |


## ⚙️ Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from IPython.display import Image, display

# Paths
BASE    = Path().resolve().parent
RAW     = BASE / "data" / "raw"
CHARTS  = BASE / "reports" / "charts"

# Style
plt.rcParams.update({
    "figure.facecolor": "#0f1117", "axes.facecolor": "#1a1d2e",
    "axes.edgecolor": "#444466", "axes.labelcolor": "#ccccdd",
    "xtick.color": "#aaaacc", "ytick.color": "#aaaacc",
    "text.color": "#ccccdd", "grid.color": "#2a2d3e",
    "grid.alpha": 0.5, "axes.titlesize": 13,
})
COLORS = ["#6c8ebf","#82b366","#d6a91a","#e07b54","#9b72cf",
          "#4db6ac","#e57373","#81c784","#ffb74d","#64b5f6"]

print("✅ Libraries loaded!")

In [ ]:
# Load all datasets
nav      = pd.read_csv(RAW / "02_nav_history.csv", parse_dates=["date"])
funds    = pd.read_csv(RAW / "01_fund_master.csv")
aum      = pd.read_csv(RAW / "03_aum_by_fund_house.csv", parse_dates=["date"])
sip      = pd.read_csv(RAW / "04_monthly_sip_inflows.csv")
cat      = pd.read_csv(RAW / "05_category_inflows.csv")
folio    = pd.read_csv(RAW / "06_industry_folio_count.csv")
txn      = pd.read_csv(RAW / "08_investor_transactions.csv", parse_dates=["transaction_date"])
holdings = pd.read_csv(RAW / "09_portfolio_holdings.csv")
bench    = pd.read_csv(RAW / "10_benchmark_indices.csv", parse_dates=["date"])
perf     = pd.read_csv(RAW / "07_scheme_performance.csv")
nav      = nav.merge(funds[["amfi_code","scheme_name","fund_house","sub_category"]], on="amfi_code", how="left")
sip["month_dt"] = pd.to_datetime(sip["month"])
folio["month_dt"] = pd.to_datetime(folio["month"])

print(f"nav_history      : {len(nav):,} rows")
print(f"transactions     : {len(txn):,} rows")
print(f"benchmark_indices: {len(bench):,} rows")
print("✅ All datasets loaded!")

---
## 📈 Chart 1: NAV Trend Analysis (2022–2026)
**Insight 1:** Large Cap Direct funds showed strong bull run in 2023, gaining 20–35% on average, followed by a consolidation phase in mid-2024.


In [ ]:
display(Image(str(CHARTS / "01_nav_trend.png"), width=900))

In [ ]:
# Interactive Plotly version
large_direct = funds[(funds["sub_category"]=="Large Cap") & (funds["plan"]=="Direct")]["amfi_code"].tolist()
nav_lc = nav[nav["amfi_code"].isin(large_direct[:6])]

fig = go.Figure()
for code, grp in nav_lc.groupby("amfi_code"):
    name = grp["scheme_name"].iloc[0].split("-")[0].strip()
    fig.add_trace(go.Scatter(x=grp["date"], y=grp["nav"], mode="lines", name=name, line=dict(width=1.5)))

fig.add_vrect(x0="2023-01-01", x1="2023-12-31", fillcolor="green", opacity=0.08,
              annotation_text="2023 Bull Run", annotation_position="top left")
fig.add_vrect(x0="2024-06-01", x1="2024-10-31", fillcolor="red", opacity=0.08,
              annotation_text="2024 Correction", annotation_position="top left")
fig.update_layout(title="Interactive NAV Trend — Large Cap Direct Funds",
                  xaxis_title="Date", yaxis_title="NAV (₹)",
                  template="plotly_dark", height=500)
fig.show()

---
## 🏦 Chart 2: AUM Growth by Fund House (2022–2025)
**Insight 2:** SBI Mutual Fund dominates with ₹12.5L Cr AUM, nearly double its 2022 base, driven by strong retail inflows and brand trust.


In [ ]:
display(Image(str(CHARTS / "02_aum_growth.png"), width=900))

---
## 💰 Chart 3: SIP Inflow Time-Series (Jan 2022 – Dec 2025)
**Insight 3:** Monthly SIP inflows grew 169% from ₹11,517 Cr (Jan 2022) to an all-time high of ₹31,002 Cr (Dec 2025), reflecting growing retail investor participation.


In [ ]:
display(Image(str(CHARTS / "03_sip_inflow.png"), width=900))

In [ ]:
# Interactive Plotly version
fig = go.Figure()
fig.add_trace(go.Scatter(x=sip["month_dt"], y=sip["sip_inflow_crore"],
    mode="lines+markers", fill="tozeroy", line=dict(color="#6c8ebf", width=2),
    fillcolor="rgba(108,142,191,0.2)", name="SIP Inflow"))

ath_idx = sip["sip_inflow_crore"].idxmax()
fig.add_annotation(x=sip.loc[ath_idx,"month_dt"], y=sip.loc[ath_idx,"sip_inflow_crore"],
    text=f"ATH ₹{sip.loc[ath_idx,'sip_inflow_crore']:,.0f} Cr",
    showarrow=True, arrowhead=2, arrowcolor="#ffb74d", font=dict(color="#ffb74d"))

fig.update_layout(title="Monthly SIP Inflows — ATH ₹31,002 Cr (Dec 2025)",
    xaxis_title="Month", yaxis_title="SIP Inflow (₹ Crore)",
    template="plotly_dark", height=450)
fig.show()

---
## 🔥 Chart 4: Category Inflow Heatmap
**Insight 4:** Large Cap and Mid Cap funds consistently attract the highest net inflows, while Debt/Liquid categories see redemption pressure during equity bull markets.


In [ ]:
display(Image(str(CHARTS / "04_category_heatmap.png"), width=1100))

---
## 👥 Chart 5: Investor Demographics
**Insight 5:** The 26–45 age group contributes over 55% of SIP transactions, while investors aged 56+ show the highest average SIP ticket size, indicating wealth accumulation behavior.


In [ ]:
display(Image(str(CHARTS / "05_demographics.png"), width=950))

In [ ]:
# Demographics summary
print("Age Group Distribution:")
print(txn["age_group"].value_counts().to_string())
print(f"\nGender Split:")
print(txn["gender"].value_counts().to_string())
print(f"\nAvg SIP Amount by Age Group:")
sip_txn = txn[txn["transaction_type"]=="SIP"]
print(sip_txn.groupby("age_group")["amount_inr"].mean().round(0).to_string())

---
## 🗺️ Chart 6: Geographic Distribution
**Insight 6:** Maharashtra, Karnataka, and Delhi NCR account for ~45% of total SIP investment amounts, while T30 cities contribute 70%+ of inflows despite B30 penetration efforts.


In [ ]:
display(Image(str(CHARTS / "06_geography.png"), width=950))

---
## 📂 Chart 7: Folio Count Growth
**Insight 7:** Total industry folios doubled from 13.26 Crore (Jan 2022) to 26.12 Crore (Dec 2025), indicating successful financial inclusion and first-time investor onboarding.


In [ ]:
display(Image(str(CHARTS / "07_folio_growth.png"), width=900))

---
## 🔗 Chart 8: NAV Return Correlation Matrix
**Insight 8:** Large Cap funds show high inter-correlation (0.85–0.95), suggesting limited diversification benefit within the same category — investors should mix categories for true diversification.


In [ ]:
display(Image(str(CHARTS / "08_correlation_matrix.png"), width=800))

---
## 🍩 Chart 9: Sector Allocation Donut
**Insight 9:** Banking & Finance dominates equity fund portfolios at ~28% weight, followed by IT (~18%) and Consumer Goods (~12%), reflecting India's economic growth drivers.


In [ ]:
display(Image(str(CHARTS / "09_sector_donut.png"), width=750))

---
## 🏆 Chart 10: Fund Performance Comparison
**Insight 10:** Direct plans consistently outperform Regular plans by 1–2% annually purely due to lower expense ratios, confirming that plan selection is critical for long-term wealth creation.


In [ ]:
display(Image(str(CHARTS / "10_performance_comparison.png"), width=900))

---
## 📊 Additional Charts (11–15)

### Chart 11: Sharpe Ratio vs 3-Year Return

In [ ]:
display(Image(str(CHARTS / '11_sharpe_ratio_vs.png'), width=900))

### Chart 12: Expense Ratio Analysis

In [ ]:
display(Image(str(CHARTS / '12_expense_ratio_analysis.png'), width=900))

### Chart 13: Monthly Transaction Volume by Type

In [ ]:
display(Image(str(CHARTS / '13_monthly_transaction_volume.png'), width=900))

### Chart 14: Fund Distribution by Risk Category

In [ ]:
display(Image(str(CHARTS / '14_fund_distribution_by.png'), width=900))

### Chart 15: Normalised NAV vs NIFTY 50 Benchmark

In [ ]:
display(Image(str(CHARTS / '15_normalised_nav_vs.png'), width=900))

---
## 📝 10 Key EDA Findings Summary

| # | Finding | Chart Ref |
|---|---------|-----------|
| 1 | Large Cap Direct funds gained 20–35% in 2023 bull run | Chart 1 |
| 2 | SBI MF dominates with ₹12.5L Cr AUM — 2× growth since 2022 | Chart 2 |
| 3 | SIP inflows hit all-time high of ₹31,002 Cr in Dec 2025 | Chart 3 |
| 4 | Large Cap & Mid Cap categories attract highest consistent inflows | Chart 4 |
| 5 | 26–45 age group drives 55%+ of SIP transactions | Chart 5 |
| 6 | Top 3 states (MH, KA, DL) account for ~45% of SIP investment | Chart 6 |
| 7 | Folios doubled from 13.26 Cr to 26.12 Cr in 4 years | Chart 7 |
| 8 | Large Cap funds are highly correlated (0.85–0.95) — low diversification | Chart 8 |
| 9 | Banking & Finance holds ~28% weight in equity fund portfolios | Chart 9 |
| 10 | Direct plans beat Regular plans by 1–2% annually on returns | Chart 10 |
